# Study Planner — Priority Model Training

Trains the model behind `POST /study-planner/predict-priority`, using the clean table from `dataset_preprocessing.ipynb`.

**Part A — model selection:** 8 classifiers are compared on the same split. XGBoost wins.

**Part B — deadline-constrained final model:** the plain XGBoost winner can rank a task due *tomorrow* **lower** than the same task due in a month. The final model fixes this inside training: it is an ordinal model made of two binary XGBoost classifiers, each constrained so that priority never rises as the deadline moves further away.

**Part C — verification and saving:** leakage guard, deadline sweep, SHAP, consistency with the deployed model, and save.

> Running this notebook **does not** replace the live model unless you set `DEPLOY_TO_APP = True` below.

## 1. Setup

In [1]:
import sys
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, f1_score,
                             roc_auc_score, roc_curve)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")


In [2]:
from pathlib import Path


def find_backend_dir(start=None):
    """Walks up from the notebook's folder to backend/ (the folder holding app/ and ml_scripts/)."""
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "app").is_dir() and (p / "ml_scripts").is_dir():
            return p
    raise FileNotFoundError("Open this notebook from inside the backend/ folder tree.")


BACKEND_DIR = find_backend_dir()
DATASETS_DIR = BACKEND_DIR / "datasets" / "study-planner"
OUTPUTS_DIR = BACKEND_DIR / "ml_scripts" / "study-planner" / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_CSV_PATH = OUTPUTS_DIR / "oulad_task_level_leakage_free.csv"
FIGURES_DIR = OUTPUTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42


def save_fig(name):
    """Charts are saved as PNG files (shown in the markdown cell below each one) rather than
    embedded in the notebook, which keeps the .ipynb small enough to diff and review."""
    path = FIGURES_DIR / f"{name}.png"
    plt.savefig(path, dpi=110, bbox_inches="tight")
    plt.close("all")
    print("Saved figure:", path.relative_to(BACKEND_DIR))


print("Backend dir :", BACKEND_DIR)
print("Datasets dir:", DATASETS_DIR)

# The final model class must come from its deployed location: joblib stores the class's
# import path, and the API loads app.services.study_planner.ordinal_priority_model.
for p in (BACKEND_DIR, BACKEND_DIR / "ml_scripts" / "study-planner"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
from app.services.study_planner.ordinal_priority_model import CLASS_ORDER, OrdinalPriorityModel
from leakage_guard import LEAKAGE_SUSPECT_THRESHOLD, find_leakage_suspects, is_leakage_suspect

MODELS_DIR = BACKEND_DIR / "trained-models" / "stuyd-planner"
DEPLOYED_MODEL_PATH = BACKEND_DIR / "app" / "models" / "study_planner" / "priority_model.joblib"

DEPLOY_TO_APP = False  # True = also overwrite the live model the API loads
print("CLASS_ORDER:", CLASS_ORDER)

Backend dir : C:\Users\Chethana Methsithum\Desktop\Smart-Uni-Guide\backend
Datasets dir: C:\Users\Chethana Methsithum\Desktop\Smart-Uni-Guide\backend\datasets\study-planner
CLASS_ORDER: ['Low', 'Medium', 'High']


## 2. Load the clean dataset

In [3]:
df = pd.read_csv(CLEAN_CSV_PATH)
feature_cols = [c for c in df.columns if c != "Priority_Label"]
assert feature_cols[0] == "date", "`date` must be feature index 0 (the deadline constraint relies on it)."
assert not {"score", "date_submitted", "submitted_late", "final_result"} & set(feature_cols), "Leakage column found!"

print(f"{df.shape[0]:,} rows, {len(feature_cols)} features")
print(feature_cols)
df["Priority_Label"].value_counts(normalize=True).round(3)

170,874 rows, 13 features
['date', 'weight', 'num_of_prev_attempts', 'studied_credits', 'module_presentation_length', 'date_registration', 'prior_avg_score', 'avg_weekly_clicks', 'clicks_trend', 'active_weeks_ratio', 'has_vle_activity', 'assessment_type_enc', 'code_module_enc']


Priority_Label
Medium    0.369
High      0.354
Low       0.277
Name: proportion, dtype: float64

## 3. Train/test split, class balance, imputation, scaling

* **80/20 stratified split**, `random_state=42` (the same split is reused for every model, including the final one).
* **SMOTE** only if the smallest class is under 15% of the training set. Here it isn't, so it is skipped.
* `date_registration` NaNs are median-imputed for the scikit-learn models. The median comes from the full table, as in the original run that produced the report numbers (it affects only a handful of rows).
* **StandardScaler** is fit on the training set only and used for the distance/linear models (LR, SVM, KNN, NB).

In [4]:
X_raw = df[feature_cols]
y = df["Priority_Label"]
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

median_reg = df["date_registration"].median()
X_train = X_train_raw.fillna({"date_registration": median_reg})
X_test = X_test_raw.fillna({"date_registration": median_reg})
print(f"Train {X_train.shape}, Test {X_test.shape}; imputed date_registration NaNs with median {median_reg}")

SMOTE_THRESHOLD = 0.15
min_class_frac = y_train.value_counts(normalize=True).min()
if min_class_frac < SMOTE_THRESHOLD:
    X_train, y_train = SMOTE(random_state=RANDOM_STATE).fit_resample(X_train, y_train)
    print(f"Smallest class {min_class_frac:.1%} < {SMOTE_THRESHOLD:.0%} -> SMOTE applied to the training set")
else:
    print(f"Smallest class {min_class_frac:.1%} >= {SMOTE_THRESHOLD:.0%} -> balanced enough, SMOTE skipped")

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_cols, index=X_test.index)

Train (136699, 13), Test (34175, 13); imputed date_registration NaNs with median -53.0
Smallest class 27.7% >= 15% -> balanced enough, SMOTE skipped


# Part A — Model selection

## 4. Baseline: Logistic Regression

In [5]:
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)
lr_proba = lr.predict_proba(X_test_scaled)[:, [list(lr.classes_).index(c) for c in CLASS_ORDER]]

print(f"Accuracy {accuracy_score(y_test, lr_pred):.4f} | Weighted F1 {f1_score(y_test, lr_pred, average='weighted'):.4f}")
print(classification_report(y_test, lr_pred, labels=CLASS_ORDER))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(lr, X_train_scaled, y_train, cv=cv, scoring="f1_weighted")
print(f"5-fold CV weighted F1: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")

Accuracy 0.6731 | Weighted F1 0.6710
              precision    recall  f1-score   support

         Low       0.66      0.81      0.73      9474
      Medium       0.64      0.63      0.64     12597
        High       0.73      0.61      0.66     12104

    accuracy                           0.67     34175
   macro avg       0.68      0.68      0.68     34175
weighted avg       0.68      0.67      0.67     34175



5-fold CV weighted F1: 0.6706 +/- 0.0033


In [6]:
y_test_bin = pd.get_dummies(y_test)[CLASS_ORDER].values
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, lr_pred, labels=CLASS_ORDER)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER, ax=axes[0])
axes[0].set(title="Logistic Regression - confusion matrix", xlabel="Predicted", ylabel="True")

for i, c in enumerate(CLASS_ORDER):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], lr_proba[:, i])
    axes[1].plot(fpr, tpr, label=f"{c} (AUC = {roc_auc_score(y_test_bin[:, i], lr_proba[:, i]):.3f})")
axes[1].plot([0, 1], [0, 1], "k--", label="Chance")
axes[1].set(title="Logistic Regression - ROC (one-vs-rest)", xlabel="False positive rate", ylabel="True positive rate")
axes[1].legend(loc="lower right")
plt.tight_layout()
save_fig("train_lr_confusion_roc")

Saved figure: ml_scripts\study-planner\outputs\figures\train_lr_confusion_roc.png


![Logistic Regression confusion matrix and ROC](outputs/figures/train_lr_confusion_roc.png)

## 5. Compare 8 classifiers

Tree models use the unscaled features; SVM, KNN and Naive Bayes use the scaled ones. XGBoost needs numeric labels, so a `LabelEncoder` maps Low/Medium/High to 0/1/2.

*SVM on ~137k rows is the slow one (several minutes).* Its `predict()` doesn't depend on `probability=True`, so that option is left off.

In [7]:
xgb_label_encoder = LabelEncoder().fit(CLASS_ORDER)

models = {
    "Random Forest": (RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_STATE), "raw"),
    "Gradient Boosting": (GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=RANDOM_STATE), "raw"),
    "XGBoost": (XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                              random_state=RANDOM_STATE, eval_metric="mlogloss"), "raw"),
    "Decision Tree": (DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE), "raw"),
    "SVM RBF": (SVC(random_state=RANDOM_STATE), "scaled"),
    "KNN": (KNeighborsClassifier(n_neighbors=7), "scaled"),
    "Naive Bayes": (GaussianNB(), "scaled"),
}

trained = {"Logistic Regression": lr}
predictions = {"Logistic Regression": lr_pred}
for name, (model, kind) in models.items():
    Xtr, Xte = (X_train, X_test) if kind == "raw" else (X_train_scaled, X_test_scaled)
    if name == "XGBoost":
        model.fit(Xtr, xgb_label_encoder.transform(y_train))
        predictions[name] = xgb_label_encoder.inverse_transform(model.predict(Xte))
    else:
        model.fit(Xtr, y_train)
        predictions[name] = model.predict(Xte)
    trained[name] = model
    print(f"trained {name}")

trained Random Forest


trained Gradient Boosting


trained XGBoost


trained Decision Tree


trained SVM RBF


trained KNN
trained Naive Bayes


In [8]:
rows = []
for name, pred in predictions.items():
    report = classification_report(y_test, pred, labels=CLASS_ORDER, output_dict=True)
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Weighted F1": f1_score(y_test, pred, average="weighted"),
        "High-Priority Precision": report["High"]["precision"],
        "High-Priority Recall": report["High"]["recall"],
        "High-Priority F1": report["High"]["f1-score"],
    })
comparison_df = pd.DataFrame(rows).sort_values("Weighted F1", ascending=False).reset_index(drop=True)
comparison_df.round(3)

,Model,Accuracy,Weighted F1,High-Priority Precision,High-Priority Recall,High-Priority F1
0,XGBoost,0.757,0.757,0.891,0.677,0.769
1,Gradient Boosting,0.753,0.753,0.895,0.666,0.764
2,Decision Tree,0.741,0.741,0.894,0.647,0.751
3,SVM RBF,0.740,0.739,0.917,0.621,0.740
4,Random Forest,0.739,0.738,0.922,0.623,0.743
5,KNN,0.729,0.729,0.806,0.693,0.745
6,Logistic Regression,0.673,0.671,0.725,0.609,0.662
7,Naive Bayes,0.612,0.608,0.585,0.717,0.645


In [9]:
ax = comparison_df.set_index("Model")[["Accuracy", "Weighted F1", "High-Priority F1"]].plot(kind="bar", figsize=(11, 5))
ax.set(title="Model comparison", ylabel="Score", ylim=(0.4, 0.85))
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
save_fig("train_model_comparison")

Saved figure: ml_scripts\study-planner\outputs\figures\train_model_comparison.png


![Model comparison](outputs/figures/train_model_comparison.png)

### Leakage guard

The label comes from real outcomes that are **not** in the features, so no honest model should reach ~97%+. If one does, the label is probably still recoverable from the inputs. This is the same check the backend test suite runs.

In [10]:
suspects = find_leakage_suspects(comparison_df, LEAKAGE_SUSPECT_THRESHOLD)
if suspects.empty:
    print(f"No model exceeds {LEAKAGE_SUSPECT_THRESHOLD:.0%} accuracy/F1 - consistent with a genuine, non-trivial task.")
else:
    print(f"WARNING - possible leakage: {suspects['Model'].tolist()}")

No model exceeds 97% accuracy/F1 - consistent with a genuine, non-trivial task.


In [11]:
best_name = comparison_df.iloc[0]["Model"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
lr_imp = pd.Series(np.abs(lr.coef_).mean(axis=0), index=feature_cols).sort_values()
axes[0].barh(lr_imp.index, lr_imp.values, color="steelblue")
axes[0].set_title("Logistic Regression (mean |coefficient|)")
tree_imp = pd.Series(trained[best_name].feature_importances_, index=feature_cols).sort_values()
axes[1].barh(tree_imp.index, tree_imp.values, color="darkorange")
axes[1].set_title(f"{best_name} (feature importance)")
plt.tight_layout()
save_fig("train_feature_importance")
print(f"Best model by weighted F1: {best_name}")

Saved figure: ml_scripts\study-planner\outputs\figures\train_feature_importance.png
Best model by weighted F1: XGBoost


![Feature importance](outputs/figures/train_feature_importance.png)

**Result:** XGBoost has the best weighted F1. Logistic Regression is clearly behind, which shows the relationship is **non-linear**: a high weight is only risky together with weak prior performance, and tree ensembles capture interactions like that. Naive Bayes has the highest High-priority *recall* but poor precision, so it would over-warn students.

# Part B — Deadline-constrained final model

## 6. The problem with the plain XGBoost winner

In OULAD, `date` is "the day of the module the deadline falls on", **not** "days remaining". The app maps days remaining onto that range (12 = due now … 261 = far away), but nothing in the plain model forces priority to fall as `date` rises.

Two otherwise-identical tasks, one due tomorrow and one due in 32 days:

In [12]:
base_row = {
    "weight": 20.0, "num_of_prev_attempts": 0, "studied_credits": 60, "module_presentation_length": 240,
    "date_registration": -30, "prior_avg_score": 65, "avg_weekly_clicks": 15, "clicks_trend": 0,
    "active_weeks_ratio": 0.5, "has_vle_activity": 1, "assessment_type_enc": 2, "code_module_enc": 0,
}
real_tasks = pd.DataFrame(
    [{**base_row, "date": 13.383333333333333}, {**base_row, "date": 56.266666666666666}],
    index=["due tomorrow", "due in 32 days"],
)[feature_cols]

xgb_plain = trained["XGBoost"]
plain_proba = xgb_plain.predict_proba(real_tasks)
pd.DataFrame({
    "date feature": real_tasks["date"].round(2),
    "predicted": xgb_label_encoder.inverse_transform(plain_proba.argmax(axis=1)),
    "confidence": plain_proba.max(axis=1).round(3),
})

,date feature,predicted,confidence
due tomorrow,13.38,Medium,0.504
due in 32 days,56.27,High,0.546


The task due **tomorrow** gets a *lower* priority than the one due in a month. That is backwards.

## 7. Why an ordinal model, not one constrained multi-class model

XGBoost can enforce a monotonic constraint on a feature, but a multi-class model applies the **same** sign to every class's trees. Here we need **P(High) to fall** and **P(Low) to rise** as `date` grows, which is opposite directions.

So priority is treated as **ordinal** (Low < Medium < High) and split into two binary problems, both with the same `date` constraint (`-1` = decreasing):

| Sub-model | Predicts |
|---|---|
| `model_medium` | P(priority ≥ Medium) |
| `model_high` | P(priority ≥ High) |

`OrdinalPriorityModel` combines them: **P(Low) = 1 − P(≥Medium)**, **P(Medium) = P(≥Medium) − P(≥High)**, **P(High) = P(≥High)**.

The constraint is enforced *inside training* instead of as an `if deadline < N days` rule afterwards. That keeps the model and its SHAP explanations consistent, and the guarantee holds for every input, not just the cases someone tested.

In [13]:
DATE_IDX = feature_cols.index("date")
monotone_constraints = tuple(-1 if i == DATE_IDX else 0 for i in range(len(feature_cols)))
print("Constraint per feature:", dict(zip(feature_cols, monotone_constraints)))

y_train_ge_medium = (y_train != "Low").astype(int)
y_train_ge_high = (y_train == "High").astype(int)
print(f"P(>=Medium) positive share: {y_train_ge_medium.mean():.3f} | P(>=High) positive share: {y_train_ge_high.mean():.3f}")

xgb_kwargs = dict(
    n_estimators=300, max_depth=6, learning_rate=0.1, random_state=RANDOM_STATE, eval_metric="logloss",
    monotone_constraints=monotone_constraints, tree_method="hist",
)
# Trained on the un-imputed features: XGBoost handles the few missing date_registration values natively.
model_medium = XGBClassifier(**xgb_kwargs).fit(X_train_raw, y_train_ge_medium)
model_high = XGBClassifier(**xgb_kwargs).fit(X_train_raw, y_train_ge_high)
final_model = OrdinalPriorityModel(model_medium, model_high)
print("Trained the deadline-constrained ordinal model.")

Constraint per feature: {'date': -1, 'weight': 0, 'num_of_prev_attempts': 0, 'studied_credits': 0, 'module_presentation_length': 0, 'date_registration': 0, 'prior_avg_score': 0, 'avg_weekly_clicks': 0, 'clicks_trend': 0, 'active_weeks_ratio': 0, 'has_vle_activity': 0, 'assessment_type_enc': 0, 'code_module_enc': 0}
P(>=Medium) positive share: 0.723 | P(>=High) positive share: 0.354


Trained the deadline-constrained ordinal model.


## 8. Plain vs constrained — same test set

In [14]:
final_pred = np.array(CLASS_ORDER)[final_model.predict(X_test_raw)]
results = []
for name, pred in [("XGBoost (plain, unconstrained)", predictions["XGBoost"]),
                   ("XGBoost ordinal (deadline-constrained)", final_pred)]:
    report = classification_report(y_test, pred, labels=CLASS_ORDER, output_dict=True)
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Weighted F1": f1_score(y_test, pred, average="weighted"),
        **{f"{c} precision": report[c]["precision"] for c in CLASS_ORDER},
        **{f"{c} recall": report[c]["recall"] for c in CLASS_ORDER},
    })
pd.DataFrame(results).set_index("Model").T.round(4)

Model,"XGBoost (plain, unconstrained)",XGBoost ordinal (deadline-constrained)
Accuracy,0.7573,0.7539
Weighted F1,0.7574,0.7539
Low precision,0.7071,0.7037
Medium precision,0.7090,0.7058
High precision,0.8908,0.8879
Low recall,0.8699,0.8718
Medium recall,0.7499,0.7436
High recall,0.6769,0.6725


In [15]:
print(classification_report(y_test, final_pred, labels=CLASS_ORDER))
cm = confusion_matrix(y_test, final_pred, labels=CLASS_ORDER)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER)
plt.title("Final model - confusion matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
save_fig("train_final_confusion")

acc, f1w = accuracy_score(y_test, final_pred), f1_score(y_test, final_pred, average="weighted")
print("Leakage guard:", "SUSPICIOUS" if is_leakage_suspect(acc, f1w) else f"OK (below {LEAKAGE_SUSPECT_THRESHOLD:.0%})")

              precision    recall  f1-score   support

         Low       0.70      0.87      0.78      9474
      Medium       0.71      0.74      0.72     12597
        High       0.89      0.67      0.77     12104

    accuracy                           0.75     34175
   macro avg       0.77      0.76      0.76     34175
weighted avg       0.77      0.75      0.75     34175

Saved figure: ml_scripts\study-planner\outputs\figures\train_final_confusion.png
Leakage guard: OK (below 97%)


![Final model confusion matrix](outputs/figures/train_final_confusion.png)

The constraint costs well under half a percentage point of accuracy, and it removes an error a student would notice straight away.

# Part C — Verification and saving

## 9. The two real tasks again

In [16]:
final_proba = final_model.predict_proba(real_tasks)
pd.DataFrame({
    "plain XGBoost": xgb_label_encoder.inverse_transform(plain_proba.argmax(axis=1)),
    "constrained": np.array(CLASS_ORDER)[final_proba.argmax(axis=1)],
    "constrained P(High)": final_proba[:, 2].round(3),
}, index=real_tasks.index)

,plain XGBoost,constrained,constrained P(High)
due tomorrow,Medium,High,0.882
due in 32 days,High,High,0.837


## 10. Deadline sweep over the full trained range

`date` is swept from 12 to 261 (30 points) with every other feature fixed. The constrained model must give a **non-increasing P(High)** and a **non-decreasing P(Low)** at every step.

In [17]:
sweep_dates = np.linspace(12, 261, 30)
sweep_rows = pd.DataFrame([{**base_row, "date": d} for d in sweep_dates])[feature_cols]
plain_sweep = xgb_plain.predict_proba(sweep_rows)  # columns follow xgb_label_encoder.classes_ (alphabetical)
final_sweep = final_model.predict_proba(sweep_rows)  # columns follow CLASS_ORDER

high_ok = bool(np.all(np.diff(final_sweep[:, 2]) <= 1e-9))
low_ok = bool(np.all(np.diff(final_sweep[:, 0]) >= -1e-9))
print(f"P(High) never rises as the deadline moves away: {high_ok}")
print(f"P(Low)  never falls as the deadline moves away: {low_ok}")

fig, ax = plt.subplots(figsize=(10, 5.5))
colors = {"Low": "#30a46c", "Medium": "#f5a524", "High": "#e5484d"}
for i, c in enumerate(CLASS_ORDER):
    ax.plot(sweep_dates, final_sweep[:, i], color=colors[c], lw=2.2, label=f"Constrained P({c})")
for i, c in enumerate(xgb_label_encoder.classes_):
    ax.plot(sweep_dates, plain_sweep[:, i], color=colors[c], ls="--", alpha=0.55, label=f"Plain P({c})")
ax.set(xlabel="date feature (12 = due now ... 261 = far away)", ylabel="Predicted probability",
       title="Priority probability vs deadline distance")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
save_fig("train_deadline_sweep")

P(High) never rises as the deadline moves away: True
P(Low)  never falls as the deadline moves away: True
Saved figure: ml_scripts\study-planner\outputs\figures\train_deadline_sweep.png


![Deadline sweep](outputs/figures/train_deadline_sweep.png)

## 11. Explainability — SHAP

SHAP needs a real tree booster, so the explainer is built on each binary sub-model. The same approach is used live in `explain_service.py`. The chart below shows global feature impact on **P(priority ≥ High)** over a 2,000-row test sample.

In [18]:
import shap

sample = X_test_raw.sample(2000, random_state=RANDOM_STATE)
shap_values = shap.TreeExplainer(model_high).shap_values(sample)
shap.summary_plot(shap_values, sample, plot_type="bar", show=False)
plt.title("Global feature impact on P(priority >= High)")
plt.tight_layout()
save_fig("train_shap_importance")

mean_abs = pd.Series(np.abs(shap_values).mean(axis=0), index=feature_cols).sort_values(ascending=False)
print("Top 5 drivers:", ", ".join(f"{k} ({v:.3f})" for k, v in mean_abs.head(5).items()))

Saved figure: ml_scripts\study-planner\outputs\figures\train_shap_importance.png
Top 5 drivers: assessment_type_enc (1.572), code_module_enc (1.086), date (0.536), avg_weekly_clicks (0.463), prior_avg_score (0.449)


![SHAP global feature impact](outputs/figures/train_shap_importance.png)

## 12. Consistency with the deployed model

In [19]:
if DEPLOYED_MODEL_PATH.exists():
    deployed = joblib.load(DEPLOYED_MODEL_PATH)
    deployed_pred = np.array(CLASS_ORDER)[deployed.predict(X_test_raw)]
    agreement = (deployed_pred == final_pred).mean()
    print(f"{type(deployed).__name__} (deployed) vs this notebook's model: {agreement:.2%} identical test-set predictions")
else:
    print("No deployed model found at", DEPLOYED_MODEL_PATH)

OrdinalPriorityModel (deployed) vs this notebook's model: 100.00% identical test-set predictions


## 13. Save

* Always saved: `trained-models/stuyd-planner/priority_model_constrained.joblib`
* Copied to `app/models/study_planner/priority_model.joblib` (the file the API loads) **only** if `DEPLOY_TO_APP = True`. Restart the backend afterwards.

The label encoders are unchanged (same alphabetical mapping), so `label_encoders.joblib` doesn't need re-saving.

In [20]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
out_path = MODELS_DIR / "priority_model_constrained.joblib"
joblib.dump(final_model, out_path)
print("Saved:", out_path)

if DEPLOY_TO_APP:
    joblib.dump(final_model, DEPLOYED_MODEL_PATH)
    print("Deployed to:", DEPLOYED_MODEL_PATH, "- restart the backend to load it.")
else:
    print("DEPLOY_TO_APP is False - the live API model was not changed.")

Saved: C:\Users\Chethana Methsithum\Desktop\Smart-Uni-Guide\backend\trained-models\stuyd-planner\priority_model_constrained.joblib
DEPLOY_TO_APP is False - the live API model was not changed.


## Summary

| Step | Outcome |
|---|---|
| Model selection | 8 classifiers compared. **XGBoost** had the best weighted F1, and tree ensembles beat linear models because of the non-linear interactions |
| Leakage guard | No model is near 97%, so the outcome-based label is a genuine prediction task |
| Problem found | The plain XGBoost could rank a nearer deadline **lower** |
| Fix | **Ordinal model of two deadline-constrained binary XGBoost classifiers** (`OrdinalPriorityModel`) |
| Cost | Under half a point of accuracy / F1 |
| Guarantee | P(High) never rises and P(Low) never falls as the deadline moves away, across the whole trained range |
| Explainability | SHAP on each binary sub-model, the same approach used live by `/study-planner/explain` |